# Machine Learning Enhanced Multi-Factor Quantitative Trading (A-Shares Demo)

This notebook demonstrates the end-to-end pipeline of our quantitative trading system described in *"Machine Learning Enhanced Multi-Factor Quantitative Trading: A Cross-Sectional Portfolio Optimization Approach with Bias Correction"* ([arXiv:2507.07107](https://arxiv.org/abs/2507.07107)).

We will build a portfolio focusing on **HS300** stocks, fetch historical market data via **Baostock**, and compute our alpha features. Finally, we'll run a vectorised backtest to get a proof-of-concept result. **You do not need to change any code, just click "Run All"!**

## 1. Environment Setup

First, we install the project dependencies, including `baostock` for fetching the Chinese A-share historical data.

In [ ]:
import os

# If we are in Colab and haven't cloned the repo yet
if not os.path.exists('pyproject.toml'):
    if not os.path.exists('ml-quant-trading'):
        !git clone https://github.com/Uwater1/ml-quant-trading.git
    %cd ml-quant-trading

!pip install -e .[dev]
!pip install baostock

## 2. Fetch HS300 Stocks Data using Baostock

We use `baostock` to get the current list of HS300 (沪深300) constituent stocks. Then we download their OHLCV and trading amount over a target period. The `mlquant.data.make_panel` utility wraps this neatly into a PyTorch-based `Panel` dataclass, masking out un-tradable days automatically.

In [ ]:
import baostock as bs
import pandas as pd
import torch
from mlquant.data import make_panel

# Log in to baostock
lg = bs.login()
print('Baostock login respond error_code:'+lg.error_code)

# 1. Fetch HS300 component list
rs = bs.query_hs300_stocks()
hs300_stocks = []
while (rs.error_code == '0') & rs.next():
    hs300_stocks.append(rs.get_row_data()[1]) # index 1 is the code like 'sh.600000'

print(f"Fetched {len(hs300_stocks)} HS300 stocks.")
bs.logout()

# 2. We will just use the first 50 stocks to keep the demo fast, 
# but you can increase this to len(hs300_stocks) for the full universe.
tickers = hs300_stocks[:50]
start_date = '2023-01-01'
end_date = '2023-12-31'

print(f"Fetching historical data for {len(tickers)} stocks from {start_date} to {end_date}...")
panel = make_panel(
    source="baostock",
    tickers=tickers,
    start=start_date,
    end=end_date,
    device="cpu" # Use "cuda" if a GPU is available
)
print(f"Created Panel with shape: Dates {panel.n_dates} x Stocks {panel.n_stocks}")

## 3. Alpha Factor Engineering and Bias Correction

Modern quant systems struggle with unintended systematic biases (e.g. market cap or industry exposure). Our system incorporates multi-stage cross-sectional neutralization to transform biased signals into pure alpha factors. 

Here we'll compute a subset of our 204 hand-crafted features (derived from `features.legacy_factors`) on the GPU/CPU.

In [ ]:
from mlquant.features import compute_legacy_set

# Compute a few selected factors
selected_factors = ("best_001", "add_015", "old_042")
print(f"Computing factors: {selected_factors}")
factors, mask, names = compute_legacy_set(panel, names=selected_factors)

print(f"Computed factor tensor shape: {factors.shape} (Dates x Stocks x Factors)")

## 4. Machine Learning Model Training

We use deep learning to combine these alpha factors into a single predictive score. The framework supports MLPs, Transformers, and gradient boosting trees (XGBoost/LightGBM).

For this demo, we'll train a lightweight multi-layer perceptron (MLP) optimizing for Information Coefficient (IC) and Rank IC.

In [ ]:
from mlquant.models.nets import CrossSectionalMLP
from mlquant.models.losses import ic_loss
from mlquant.training.dataset import PanelDataset
from torch.utils.data import DataLoader
import torch.optim as optim

# Create dataset
# Target: Forward 1-day returns
targets = panel.returns.roll(shifts=-1, dims=0) # T, N
targets[-1] = 0.0 # Last day has no forward return

dataset = PanelDataset(factors, targets, mask)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize Model
model = CrossSectionalMLP(in_features=factors.shape[-1], hidden_dims=[64, 32], dropout=0.1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Training model for 5 epochs...")
model.train()
for epoch in range(5):
    total_loss = 0.0
    for X, y, m in loader:
        optimizer.zero_grad()
        preds = model(X)
        
        # Using our custom IC loss: negative cross-sectional correlation
        loss = ic_loss(preds, y, m)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/5 | Average IC Loss: {total_loss / len(loader):.4f}")

print("Training complete.")

## 5. Inference and Cross-Sectional Portfolio Optimization

Rather than predicting absolute returns, the model focuses on relative performance within the universe. This naturally hedges market risk while concentrating on security selection alpha.

We run the model over the full period to get predicted scores, then apply a cross-sectional Markowitz optimizer (with leverage and no-short constraints) to generate the target portfolio weights day by day.

In [ ]:
from mlquant.portfolio.markowitz import optimize_markowitz

model.eval()
with torch.no_grad():
    # Get daily predictions
    # X has shape [T, N, F]
    predictions = model(factors)
    
print("Generating portfolio weights...")
T, N = panel.n_dates, panel.n_stocks
weights = torch.zeros((T, N))

# Simple top-K weighting for demonstration
for t in range(T):
    day_mask = mask[t]
    if day_mask.sum() > 5:
        day_preds = predictions[t]
        day_preds[~day_mask] = -1e9
        # Buy top 5 stocks each day
        top_idx = torch.topk(day_preds, k=5).indices
        weights[t, top_idx] = 1.0 / 5.0
        
print("Portfolio weights calculated.")

## 6. Vectorised Backtest & Metrics

Finally, we run a fast vectorised backtest to evaluate the strategy's Sharpe Ratio, Maximum Drawdown, and Annual Return. Our backtester rigorously handles untradable dates (limit-ups, halts) automatically using the `panel.mask`.

In [ ]:
from mlquant.backtest.engine import VectorizedBacktest

print("Running backtest...")
# Instantiate backtester and run
engine = VectorizedBacktest(panel, weights)
results = engine.run(cost_bps=15.0) # Assume 15bps trading cost

print("\n=== Backtest Results ===")
print(f"Annualized Return : {results['annual_return'] * 100:.2f}%")
print(f"Sharpe Ratio      : {results['sharpe']:.2f}")
print(f"Max Drawdown      : {results['max_drawdown'] * 100:.2f}%")
print(f"Daily Turnover    : {results['turnover'] * 100:.2f}%")

print("\nProof-of-concept complete! To improve these results, train on a larger dataset with more factors, and utilize the full Markowitz portfolio optimizer instead of simple Top-K selection.")